# Let's start looking into building a compound generator.

This step is the same as step 7 except we will now implement the other other 4 models as criteria for reinforced learning.

**Switch environment to Python 11 with GPU (cuda). You will most likely have to run this notebook a second time due to numpy updates**

In [1]:
!python --version

Python 3.11.13


In [2]:
!pip install tensorflow==2.14.1

In [3]:
!pip install chemtsv2

In [4]:
!pip install chemprop

### Let's first create a reward system and turn it into a .py file

In [11]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_reward.py
import sys
import numpy as np
from rdkit.Chem import Descriptors
from chemtsv2.abc import Reward
from chemprop.models.model import MPNN
from chemprop import data, featurizers, models
from lightning import pytorch as pl
from rdkit import Chem
import torch


TRPV1 = '/content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/models/best-epoch=52-val_loss=0.33.ckpt'
MOR = '/content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/models/MOR_best-epoch=43-val_loss=0.35.ckpt'
DOR = '/content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/models/DOR_best-epoch=48-val_loss=0.33.ckpt'
KOR = '/content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/models/KOR_best-epoch=56-val_loss=0.41.ckpt'
NOCICEPTIN = '/content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/models/Nociceptin_best-epoch=58-val_loss=0.43.ckpt'


def predict_receptivity(mol, model_path):
  # turn mol to smiles
  smiles = []
  smile = Chem.MolToSmiles(mol)
  smiles.append(smile)

  # load model
  checkpoint_path = model_path # CHANGE THIS TO WHERE THE MODEL .CKPT FILE IS SAVED
  mpnn = MPNN.load_from_checkpoint(checkpoint_path)

  # assign data
  test_data = [data.MoleculeDatapoint.from_smi(smi) for smi in smiles]

  # set model vars
  featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
  test_dset = data.MoleculeDataset(test_data, featurizer=featurizer)
  test_loader = data.build_dataloader(test_dset, shuffle=False)

  # predict
  with torch.inference_mode():
    trainer = pl.Trainer(
        logger=None,
        enable_progress_bar=True,
        accelerator="cuda",
        devices=1
    )
    test_preds = trainer.predict(mpnn, test_loader)

  # extract only pchembl value
  return test_preds[0][0].numpy()[0]


class pchemble_reward(Reward):
  def get_objective_functions(conf):
    # we need to hook our chemprop model here to predict a pChEMBLE value
    def predict_trpv1_receptivity(mol):
      return predict_receptivity(mol, TRPV1)

    def predict_mor_receptivity(mol):
      return predict_receptivity(mol, MOR)

    def predict_dor_receptivity(mol):
      return predict_receptivity(mol, DOR)

    def predict_kor_receptivity(mol):
      return predict_receptivity(mol, KOR)

    def predict_nociceptin_receptivity(mol):
      return predict_receptivity(mol, NOCICEPTIN)

    return [predict_trpv1_receptivity, predict_mor_receptivity, predict_dor_receptivity, predict_kor_receptivity, predict_nociceptin_receptivity]

  def calc_reward_from_objective_values(values, conf):
    max_pchemble_val = 10 # lets arbitrarily set max score to 10

    # all values
    trpv1_pchembl_pred = values[0]
    mor_pchembl_pred = values[1]
    dor_pchembl_pred = values[2]
    kor_pchembl_pred = values[3]
    nociceptin_pchembl_pred = values[4]

    # positive criteria - get all favorable traits
    trpv1_score = trpv1_pchembl_pred / max_pchemble_val

    # negative criteria - get all unfavorable traits
    negative_scores = []
    mor_score = mor_pchembl_pred / max_pchemble_val
    dor_score = dor_pchembl_pred / max_pchemble_val
    kor_score = kor_pchembl_pred / max_pchemble_val
    nociceptin_score = nociceptin_pchembl_pred / max_pchemble_val

    # store positive scores in array
    positive_scores = []
    positive_scores.append(trpv1_score)

    # store negative scores in array
    negative_scores = []
    negative_scores.append(mor_score)
    negative_scores.append(dor_score)
    negative_scores.append(kor_score)
    negative_scores.append(nociceptin_score)

    positive_sum = sum(positive_scores) / len(positive_scores)
    negative_sum = sum(negative_scores) / len(negative_scores)

    sum_of_scores = positive_sum - negative_sum
    return sum_of_scores

Overwriting /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_reward.py


### Let's also create a config .yaml file

In [6]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_setting.yaml

# Basic setting
c_val: 1.0
# threshold_type: [time, generation_num]
threshold_type: generation_num
#hours: 0.01
generation_num: 500
output_dir: result/example01
model_setting:
  model_json: model/model.tf25.json
  model_weight: model/model.tf25.best.ckpt.h5
token: model/tokens.pkl
reward_setting:
  reward_module: reward.pchemble_reward
  reward_class: pchemble_reward

# Advanced setting
expansion_threshold: 0.995
simulation_num: 3
flush_threshold: -1
policy_setting:
  policy_module: policy.ucb1
  policy_class: Ucb1

# Restart setting
save_checkpoint: False
restart: False
checkpoint_file: chemtsv2.ckpt.pkl

# Filter setting
use_lipinski_filter: True
lipinski_filter:
  module: filter.lipinski_filter
  class: LipinskiFilter
  type: rule_of_5
use_radical_filter: True
radical_filter:
  module: filter.radical_filter
  class: RadicalFilter
use_pubchem_filter: True
pubchem_filter:
  module: filter.pubchem_filter
  class: PubchemFilter
use_sascore_filter: True
sascore_filter:
  module: filter.sascore_filter
  class: SascoreFilter
  threshold: 3.5
use_ring_size_filter: True
ring_size_filter:
  module: filter.ring_size_filter
  class: RingSizeFilter
  threshold: 6
use_pains_filter: False
pains_filter:
  module: filter.pains_filter
  class: PainsFilter
  type: [pains_a]
use_covalent_warhead_filter: False
covalent_warhead_filter:
  module: filter.covalent_warhead_filter
  class: CovalentWarheadFilter
include_filter_result_in_reward: False

Overwriting /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_setting.yaml


## We've now succesfully created our reward and config file for ChemTSV. Let's now start generating structures!!


In [7]:
!git clone https://github.com/molecule-generator-collection/ChemTSv2.git

fatal: destination path 'ChemTSv2' already exists and is not an empty directory.


In [12]:
# copy reward and config file into the chemtsv repo
!cp /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_reward.py /content/ChemTSv2/reward/pchemble_reward.py
!cp /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_setting.yaml /content/ChemTSv2/config/pchemble_setting.yaml

In [13]:
%cd ChemTSv2
!chemtsv2 -c config/pchemble_setting.yaml --gpu 0

Streaming output truncated to the last 5000 lines.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:00:00 • 0:00:00 0.00it/s 
Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 1, batch_size = 64)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/

In [10]:
assert False

AssertionError: 

In [14]:
# Run this cell to downlaod the best model file
from google.colab import files

results_file = !find result/example01 -name '*.csv'
results_file

# download csv file
files.download(results_file[0])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>